# Notebook 05: Comprehensive Algorithm Comparison and Benchmarking

## Overview

This notebook provides a comprehensive comparison of all pathfinding algorithms in our lab:

1. **BFS** - Breadth-First Search
2. **DFS** - Depth-First Search
3. **Dijkstra** - Optimal weighted pathfinding
4. **A*** - Dijkstra with heuristic guidance
5. **Greedy Best-First** - Heuristic-only search
6. **Bidirectional BFS** - Searches from both ends

## Learning Objectives

By the end of this notebook, you will:
1. Understand the strengths and weaknesses of each algorithm
2. Know which algorithm to use for different scenarios
3. Analyze trade-offs between optimality, completeness, and efficiency
4. Interpret benchmark results and performance metrics
5. Make informed decisions about algorithm selection

## Key Concepts

### Algorithm Properties

| Algorithm | Complete? | Optimal? | Time Complexity | Space Complexity |
|-----------|-----------|----------|-----------------|------------------|
| BFS | Yes | Yes* | O(V + E) | O(V) |
| DFS | Yes | No | O(V + E) | O(V) |
| Dijkstra | Yes | Yes | O((V+E) log V) | O(V) |
| A* | Yes | Yes** | O((V+E) log V) | O(V) |
| Greedy | Yes | No | O((V+E) log V) | O(V) |
| Bidirectional BFS | Yes | Yes* | O(V + E) | O(V) |

\* Optimal for unweighted graphs  
** Optimal if heuristic is admissible

## Setup and Imports

In [ ]:
# Add parent directory to path for imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
from src.pathfinding_lab.core.grid import Grid
from src.pathfinding_lab.core.types import MovementMode

# All algorithms
from src.pathfinding_lab.algorithms.bfs import bfs
from src.pathfinding_lab.algorithms.dfs import dfs
from src.pathfinding_lab.algorithms.dijkstra import dijkstra
from src.pathfinding_lab.algorithms.astar import astar
from src.pathfinding_lab.algorithms.greedy_best_first import greedy_best_first
from src.pathfinding_lab.algorithms.bidirectional_bfs import bidirectional_bfs

# Heuristics
from src.pathfinding_lab.heuristics.manhattan import manhattan_distance
from src.pathfinding_lab.heuristics.euclidean import euclidean_distance
from src.pathfinding_lab.heuristics.octile import octile_distance

# Benchmarking
from src.pathfinding_lab.metrics.benchmark import run_comparison

# Visualization
from src.pathfinding_lab.visualization.grid_plot import create_grid_plot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ Imports successful!")

## Part 1: Head-to-Head Comparison on a Standard Grid

In [ ]:
# Create a standard test grid
test_grid = Grid(width=30, height=30, obstacle_density=0.2,
                 movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)

test_start = (2, 2)
test_goal = (27, 27)
test_grid.generate_obstacles(test_start, test_goal)

print(f"Test Grid: {test_grid.width}x{test_grid.height}")
print(f"Obstacles: {len(test_grid.obstacles)} ({test_grid.obstacle_density*100:.0f}%)")
print(f"Start: {test_start}")
print(f"Goal: {test_goal}")
print(f"Movement: 8-directional (diagonals allowed)")

In [ ]:
# Run all algorithms
print("Running all algorithms...\n")

results = {}

# BFS
print("1/6 Running BFS...")
results['BFS'] = bfs(test_grid, test_start, test_goal)

# DFS
print("2/6 Running DFS...")
results['DFS'] = dfs(test_grid, test_start, test_goal)

# Dijkstra
print("3/6 Running Dijkstra...")
results['Dijkstra'] = dijkstra(test_grid, test_start, test_goal)

# A* with Octile (best for 8-directional)
print("4/6 Running A* (Octile)...")
results['A* (Octile)'] = astar(test_grid, test_start, test_goal, octile_distance)

# Greedy Best-First
print("5/6 Running Greedy Best-First...")
results['Greedy'] = greedy_best_first(test_grid, test_start, test_goal, octile_distance)

# Bidirectional BFS
print("6/6 Running Bidirectional BFS...")
results['Bi-BFS'] = bidirectional_bfs(test_grid, test_start, test_goal)

print("\n✓ All algorithms completed!")

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for name, result in results.items():
    comparison_data.append({
        'Algorithm': name,
        'Success': '✓' if result.success else '✗',
        'Path Length': result.path_length if result.success else 'N/A',
        'Path Cost': f"{result.path_cost:.3f}" if result.success else 'N/A',
        'Nodes Visited': result.nodes_visited,
        'Runtime (ms)': f"{result.runtime_ms:.2f}",
        'Optimal': 'Yes' if name in ['Dijkstra', 'A* (Octile)'] else 'No'
    })

df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPREHENSIVE ALGORITHM COMPARISON")
print("="*100)
print(df.to_string(index=False))
print("="*100)

# Find optimal cost
optimal_cost = results['Dijkstra'].path_cost
print(f"\nOptimal Path Cost: {optimal_cost:.3f}")
print(f"\nAlgorithms Finding Optimal Path: Dijkstra, A*")
print(f"Algorithms Finding Suboptimal Path: BFS, Greedy, Bi-BFS")
print(f"Algorithms Not Guaranteed Optimal: DFS")

## Part 2: Visual Comparison

In [ ]:
# Create 2x3 grid of visualizations
fig, axes = plt.subplots(2, 3, figsize=(24, 16))
axes = axes.flatten()

def plot_algorithm_result(ax, grid, start, goal, result, title):
    """Helper function to plot algorithm result on a given axis."""
    vis_grid = np.zeros((grid.height, grid.width))
    
    # Mark obstacles
    for obstacle in grid.obstacles:
        vis_grid[obstacle[0], obstacle[1]] = 1
    
    # Mark visited nodes
    if result and result.visited_order:
        for pos in result.visited_order:
            if pos != start and pos != goal:
                vis_grid[pos[0], pos[1]] = 2
    
    # Mark path
    if result and result.path:
        for pos in result.path:
            if pos != start and pos != goal:
                vis_grid[pos[0], pos[1]] = 3
    
    # Mark start and goal
    vis_grid[start[0], start[1]] = 4
    vis_grid[goal[0], goal[1]] = 5
    
    # Plot
    colors = ['white', 'black', 'lightblue', 'yellow', 'green', 'red']
    cmap = ListedColormap(colors)
    ax.imshow(vis_grid, cmap=cmap, vmin=0, vmax=5)
    
    # Grid lines
    ax.set_xticks(np.arange(-0.5, grid.width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.height, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.3)
    ax.tick_params(which='minor', size=0)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Title with stats
    stats = f"Cost: {result.path_cost:.2f}, Visited: {result.nodes_visited}, Time: {result.runtime_ms:.2f}ms"
    ax.set_title(f"{title}\n{stats}", fontsize=11, fontweight='bold', pad=10)

# Plot each algorithm
for idx, (name, result) in enumerate(results.items()):
    plot_algorithm_result(axes[idx], test_grid, test_start, test_goal, result, name)

plt.tight_layout()
plt.show()

print("\nVisual Observations:")
print("- Blue cells: Nodes visited during search")
print("- Yellow cells: Final path found")
print("- Notice how A* visits far fewer nodes than Dijkstra")
print("- Greedy is fast but may find suboptimal paths")
print("- DFS explores deeply but inefficiently")

## Part 3: Performance Metrics Analysis

In [ ]:
# Extract metrics for plotting
algorithms = list(results.keys())
nodes_visited = [results[alg].nodes_visited for alg in algorithms]
runtimes = [results[alg].runtime_ms for alg in algorithms]
path_costs = [results[alg].path_cost if results[alg].success else 0 for alg in algorithms]
path_lengths = [results[alg].path_length if results[alg].success else 0 for alg in algorithms]

# Create comprehensive performance plots
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Nodes Visited
bars1 = axes[0, 0].bar(algorithms, nodes_visited, color=sns.color_palette("husl", len(algorithms)), 
                       alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Nodes Visited', fontsize=13, fontweight='bold')
axes[0, 0].set_title('Search Efficiency\n(Lower is Better)', fontsize=14, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)
# Add value labels
for bar in bars1:
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom', fontsize=9)

# 2. Runtime
bars2 = axes[0, 1].bar(algorithms, runtimes, color=sns.color_palette("husl", len(algorithms)), 
                       alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Runtime (ms)', fontsize=13, fontweight='bold')
axes[0, 1].set_title('Computation Time\n(Lower is Better)', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)
for bar in bars2:
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 3. Path Cost
bars3 = axes[1, 0].bar(algorithms, path_costs, color=sns.color_palette("husl", len(algorithms)), 
                       alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].axhline(y=optimal_cost, color='green', linestyle='--', linewidth=2.5, label='Optimal Cost')
axes[1, 0].set_ylabel('Path Cost', fontsize=13, fontweight='bold')
axes[1, 0].set_title('Path Optimality\n(Lower is Better)', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(axis='y', alpha=0.3)
for bar in bars3:
    height = bar.get_height()
    if height > 0:
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 4. Efficiency Ratio (Normalized to Dijkstra)
dijkstra_visited = results['Dijkstra'].nodes_visited
efficiency_ratios = [v / dijkstra_visited for v in nodes_visited]
bars4 = axes[1, 1].bar(algorithms, efficiency_ratios, color=sns.color_palette("husl", len(algorithms)), 
                       alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].axhline(y=1.0, color='red', linestyle='--', linewidth=2.5, label='Dijkstra Baseline')
axes[1, 1].set_ylabel('Relative Efficiency', fontsize=13, fontweight='bold')
axes[1, 1].set_title('Efficiency vs Dijkstra\n(Lower is Better)', fontsize=14, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(axis='y', alpha=0.3)
for bar in bars4:
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}x', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Part 4: Scalability Analysis

How do algorithms perform as grid size increases?

In [ ]:
# Test across different grid sizes
grid_sizes = [10, 15, 20, 25, 30, 35, 40]
obstacle_density = 0.15

scalability_results = {alg: {'sizes': [], 'visited': [], 'time': [], 'cost': []} 
                       for alg in ['BFS', 'Dijkstra', 'A*', 'Greedy', 'Bi-BFS']}

print("Running scalability tests...\n")
print(f"{'Size':>6} | {'BFS':>8} | {'Dijkstra':>8} | {'A*':>8} | {'Greedy':>8} | {'Bi-BFS':>8}")
print("-" * 70)

for size in grid_sizes:
    # Create test grid
    grid = Grid(width=size, height=size, obstacle_density=obstacle_density,
                movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)
    start = (1, 1)
    goal = (size-2, size-2)
    grid.generate_obstacles(start, goal)
    
    # Run algorithms
    r_bfs = bfs(grid, start, goal)
    r_dijkstra = dijkstra(grid, start, goal)
    r_astar = astar(grid, start, goal, octile_distance)
    r_greedy = greedy_best_first(grid, start, goal, octile_distance)
    r_bibfs = bidirectional_bfs(grid, start, goal)
    
    # Store results
    for alg, result in [('BFS', r_bfs), ('Dijkstra', r_dijkstra), ('A*', r_astar), 
                        ('Greedy', r_greedy), ('Bi-BFS', r_bibfs)]:
        scalability_results[alg]['sizes'].append(size)
        scalability_results[alg]['visited'].append(result.nodes_visited)
        scalability_results[alg]['time'].append(result.runtime_ms)
        scalability_results[alg]['cost'].append(result.path_cost if result.success else 0)
    
    print(f"{size:>6} | {r_bfs.runtime_ms:>8.2f} | {r_dijkstra.runtime_ms:>8.2f} | "
          f"{r_astar.runtime_ms:>8.2f} | {r_greedy.runtime_ms:>8.2f} | {r_bibfs.runtime_ms:>8.2f}")

print("\n✓ Scalability tests completed!")

In [ ]:
# Plot scalability results
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Runtime vs Grid Size
for alg in ['BFS', 'Dijkstra', 'A*', 'Greedy', 'Bi-BFS']:
    axes[0].plot(scalability_results[alg]['sizes'], scalability_results[alg]['time'],
                marker='o', linewidth=2.5, markersize=8, label=alg)
axes[0].set_xlabel('Grid Size (N x N)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Runtime (ms)', fontsize=13, fontweight='bold')
axes[0].set_title('Runtime Scalability', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Nodes Visited vs Grid Size
for alg in ['BFS', 'Dijkstra', 'A*', 'Greedy', 'Bi-BFS']:
    axes[1].plot(scalability_results[alg]['sizes'], scalability_results[alg]['visited'],
                marker='s', linewidth=2.5, markersize=8, label=alg)
axes[1].set_xlabel('Grid Size (N x N)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nodes Visited', fontsize=13, fontweight='bold')
axes[1].set_title('Search Space Exploration', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nScalability Insights:")
print("- A* consistently outperforms Dijkstra as grid size increases")
print("- Bidirectional BFS is very efficient for unweighted graphs")
print("- Greedy is fastest but doesn't guarantee optimal paths")
print("- DFS omitted due to unpredictable behavior")

## Part 5: Scenario-Based Recommendations

In [ ]:
# Test algorithms on different scenarios
scenarios = [
    ("Open Space", 40, 0.05, MovementMode.EIGHT_DIRECTIONAL),
    ("Moderate Obstacles", 40, 0.20, MovementMode.EIGHT_DIRECTIONAL),
    ("Dense Obstacles", 40, 0.35, MovementMode.EIGHT_DIRECTIONAL),
    ("Small Grid", 15, 0.15, MovementMode.FOUR_DIRECTIONAL),
    ("Large Grid", 50, 0.15, MovementMode.EIGHT_DIRECTIONAL),
]

print("Testing algorithms on different scenarios...\n")

scenario_data = []

for scenario_name, size, density, movement in scenarios:
    print(f"\nScenario: {scenario_name} ({size}x{size}, {density*100:.0f}% obstacles)")
    print("-" * 80)
    
    # Create grid
    grid = Grid(width=size, height=size, obstacle_density=density,
                movement_mode=movement, random_seed=42)
    start = (1, 1)
    goal = (size-2, size-2)
    grid.generate_obstacles(start, goal)
    
    # Run algorithms
    r_dijkstra = dijkstra(grid, start, goal)
    r_astar = astar(grid, start, goal, octile_distance)
    r_greedy = greedy_best_first(grid, start, goal, octile_distance)
    
    optimal_cost = r_dijkstra.path_cost
    
    # Calculate metrics
    astar_speedup = r_dijkstra.runtime_ms / r_astar.runtime_ms if r_astar.runtime_ms > 0 else 0
    greedy_subopt = ((r_greedy.path_cost - optimal_cost) / optimal_cost * 100) if optimal_cost > 0 else 0
    
    scenario_data.append({
        'Scenario': scenario_name,
        'Dijkstra Time': f"{r_dijkstra.runtime_ms:.2f}ms",
        'A* Time': f"{r_astar.runtime_ms:.2f}ms",
        'A* Speedup': f"{astar_speedup:.2f}x",
        'Greedy Subopt': f"{greedy_subopt:.1f}%",
        'Best Choice': 'A*' if astar_speedup > 1.2 else 'Dijkstra'
    })
    
    print(f"Dijkstra: {r_dijkstra.nodes_visited:4d} nodes, {r_dijkstra.runtime_ms:6.2f}ms")
    print(f"A*:       {r_astar.nodes_visited:4d} nodes, {r_astar.runtime_ms:6.2f}ms (Speedup: {astar_speedup:.2f}x)")
    print(f"Greedy:   {r_greedy.nodes_visited:4d} nodes, {r_greedy.runtime_ms:6.2f}ms (Subopt: {greedy_subopt:.1f}%)")

# Display summary table
df_scenarios = pd.DataFrame(scenario_data)
print("\n" + "="*100)
print("SCENARIO-BASED RECOMMENDATIONS")
print("="*100)
print(df_scenarios.to_string(index=False))
print("="*100)

## Part 6: Statistical Summary and Rankings

In [ ]:
# Create ranking table
ranking_data = [
    {
        'Algorithm': 'A* (with good heuristic)',
        'Optimality': '★★★★★',
        'Speed': '★★★★★',
        'Memory': '★★★★☆',
        'Simplicity': '★★★☆☆',
        'Overall': '★★★★★',
        'Best For': 'Weighted graphs, general pathfinding'
    },
    {
        'Algorithm': 'Dijkstra',
        'Optimality': '★★★★★',
        'Speed': '★★★☆☆',
        'Memory': '★★★★☆',
        'Simplicity': '★★★★☆',
        'Overall': '★★★★☆',
        'Best For': 'When no heuristic available, all-pairs shortest path'
    },
    {
        'Algorithm': 'Bidirectional BFS',
        'Optimality': '★★★★★',
        'Speed': '★★★★☆',
        'Memory': '★★★☆☆',
        'Simplicity': '★★★☆☆',
        'Overall': '★★★★☆',
        'Best For': 'Unweighted graphs, known goal'
    },
    {
        'Algorithm': 'BFS',
        'Optimality': '★★★★☆',
        'Speed': '★★★☆☆',
        'Memory': '★★★★☆',
        'Simplicity': '★★★★★',
        'Overall': '★★★★☆',
        'Best For': 'Unweighted graphs, simple implementation'
    },
    {
        'Algorithm': 'Greedy Best-First',
        'Optimality': '★★☆☆☆',
        'Speed': '★★★★★',
        'Memory': '★★★★☆',
        'Simplicity': '★★★☆☆',
        'Overall': '★★★☆☆',
        'Best For': 'Fast approximate solutions, non-critical paths'
    },
    {
        'Algorithm': 'DFS',
        'Optimality': '★☆☆☆☆',
        'Speed': '★★☆☆☆',
        'Memory': '★★★★★',
        'Simplicity': '★★★★★',
        'Overall': '★★☆☆☆',
        'Best For': 'Maze generation, topological sorting, NOT pathfinding'
    },
]

df_ranking = pd.DataFrame(ranking_data)

print("\n" + "="*110)
print("ALGORITHM RANKINGS")
print("="*110)
print(df_ranking.to_string(index=False))
print("="*110)
print("\n★ = Star rating (more stars = better)")

## Exercise 1: Custom Benchmark

Create your own benchmark scenario and test all algorithms.

In [ ]:
# Your code here
# Create a custom grid scenario
# Run multiple algorithms
# Compare results

# Example:
# custom_grid = Grid(width=..., height=..., obstacle_density=..., movement_mode=...)
# ...

## Exercise 2: Monte Carlo Analysis

Run algorithms on multiple random grids and compute average performance.

In [ ]:
# Your code here
# Generate N random grids with different seeds
# Run A* and Dijkstra on each
# Compute mean and std deviation of metrics
# Create box plots

# ...

## Key Takeaways and Decision Guide

### Quick Decision Tree:

```
Do you need the OPTIMAL path?
├─ YES
│  ├─ Is your graph WEIGHTED (e.g., diagonal costs differ)?
│  │  ├─ YES: Use **A*** (if you have a heuristic) or **Dijkstra**
│  │  └─ NO: Use **BFS** or **Bidirectional BFS**
│  └─
└─ NO (approximate solution acceptable)
   └─ Use **Greedy Best-First** for speed
```

### Algorithm Selection Matrix:

| Scenario | Recommended Algorithm | Why |
|----------|----------------------|-----|
| Shortest path in grid (4-dir) | A* with Manhattan | Optimal, very efficient |
| Shortest path in grid (8-dir) | A* with Octile | Optimal, tight heuristic |
| No heuristic available | Dijkstra | Optimal without heuristic |
| Unweighted graph | Bidirectional BFS | Faster than regular BFS |
| Need quick approximate path | Greedy Best-First | Fast but suboptimal |
| Very large graph | A* with tight heuristic | Scales best |
| Memory constrained | DFS | Low memory, but poor paths |

### Performance Summary:

**Best Overall:** A* with appropriate heuristic
- Optimal paths (with admissible heuristic)
- Efficient exploration
- Scales well

**Best for Simplicity:** BFS
- Easy to implement
- Optimal for unweighted graphs
- Good baseline

**Best for Speed:** Greedy Best-First
- Fastest search
- Good approximations
- No optimality guarantee

**Worst for Pathfinding:** DFS
- No optimality
- Unpredictable
- Use for other graph problems instead

## Next Steps

In the next notebook (06_learned_heuristics.ipynb), we'll explore:
- Machine learning-based heuristics
- Training models to predict distances
- Comparing learned vs classical heuristics
- Understanding the trade-offs

## Further Reading

- [Pathfinding Algorithms Comparison](http://theory.stanford.edu/~amitp/GameProgramming/)
- Cormen et al., "Introduction to Algorithms" (CLRS)
- Russell & Norvig, "Artificial Intelligence: A Modern Approach"